# Обучение BERT-модели для извлечения сущностей из резюме

Ноутбук обучает NER-модель для извлечения ключевых сущностей из текста резюме.


## Установка зависимостей


In [1]:
!pip install -q transformers datasets evaluate seqeval accelerate scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00


## Импорты


In [2]:
from pathlib import Path
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

import evaluate

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [6]:
BASE_DIR = Path(".")

DATA_DIR = BASE_DIR / "data" / "raw" / "dataturks_resume_ner"
MODEL_DIR = BASE_DIR / "app" / "models" / "resume_bert_ner_model_chunked"
REPORT_DIR = BASE_DIR / "experiments" / "reports"
CHECKPOINT_DIR = BASE_DIR / "experiments" / "bert_cased_ner_checkpoints"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "train.json"

print("DATA_PATH:", DATA_PATH)
print("MODEL_DIR:", MODEL_DIR)
print("REPORT_DIR:", REPORT_DIR)

DATA_PATH: data/raw/dataturks_resume_ner/train.json
MODEL_DIR: app/models/resume_bert_ner_model_chunked
REPORT_DIR: experiments/reports


## Загрузка исходного датасета


In [7]:
raw_data = json.loads(DATA_PATH.read_text(encoding="utf-8"))

print("Type:", type(raw_data))
print("Length:", len(raw_data))
print("First item keys:", raw_data[0].keys())
print()
print("Text preview:")
print(raw_data[0]["text"][:700])
print()
print("Annotations preview:")
print(raw_data[0]["annotations"][:10])

Type: <class 'list'>
Length: 5960
First item keys: dict_keys(['text', 'annotations'])

Text preview:
Abhishek Jha Application Development Associate - Accenture  Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a  • To work for an organization which provides me the opportunity to improve my skills and knowledge for my individual and company's growth in best possible ways.  Willing to relocate to: Bangalore, Karnataka  WORK EXPERIENCE  Application Development Associate  Accenture -  November 2017 to Present  Role: Currently working on Chat-bot. Developing Backend Oracle PeopleSoft Queries for the Bot which will be triggered based on given input. Also, Training the bot for different possible utterances (Both positive and negative), which will be given as inpu

Annotations preview:
[[1296, 1622, 'SKILL'], [993, 1154, 'SKILL'], [939, 957, 'PERSON'], [883, 905, 'PERSON'], [429, 433, 'EDUCATION'], [771, 814, 'PERSON'], [727, 769, 'DESIGNATION'], [49, 58, 'CO

## Конфигурация эксперимента

В качестве базовой модели используется `bert-base-cased`.  


In [8]:
MODEL_NAME = "bert-base-cased"

LABELS_TO_KEEP = [
    "PERSON",
    "DESIGNATION",
    "COMPANY",
    "LOCATION",
    "EDUCATION",
]

N_EPOCHS = 5
MAX_EXAMPLES = None
MAX_LENGTH = 512
CHUNK_MAX_CHARS = 2000
CHUNK_OVERLAP_CHARS = 300
COMPANY_OVERSAMPLE_MULTIPLIER = 5

print("MODEL_NAME:", MODEL_NAME)
print("LABELS_TO_KEEP:", LABELS_TO_KEEP)
print("N_EPOCHS:", N_EPOCHS)
print("MAX_EXAMPLES:", MAX_EXAMPLES)
print("MAX_LENGTH:", MAX_LENGTH)
print("CHUNK_MAX_CHARS:", CHUNK_MAX_CHARS)
print("CHUNK_OVERLAP_CHARS:", CHUNK_OVERLAP_CHARS)
print("COMPANY_OVERSAMPLE_MULTIPLIER:", COMPANY_OVERSAMPLE_MULTIPLIER)


MODEL_NAME: bert-base-cased
LABELS_TO_KEEP: ['PERSON', 'DESIGNATION', 'COMPANY', 'LOCATION', 'EDUCATION']
N_EPOCHS: 5
MAX_EXAMPLES: None
MAX_LENGTH: 512
CHUNK_MAX_CHARS: 2000
CHUNK_OVERLAP_CHARS: 300
COMPANY_OVERSAMPLE_MULTIPLIER: 5


## Подготовка BIO-меток


In [9]:
label_list = ["O"]

for label in LABELS_TO_KEEP:
    label_list.append(f"B-{label}")
    label_list.append(f"I-{label}")

label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

print("Labels:")
for idx, label in id2label.items():
    print(idx, label)


Labels:
0 O
1 B-PERSON
2 I-PERSON
3 B-DESIGNATION
4 I-DESIGNATION
5 B-COMPANY
6 I-COMPANY
7 B-LOCATION
8 I-LOCATION
9 B-EDUCATION
10 I-EDUCATION


## Функции очистки текста и корректировки spans


In [10]:
def remove_surrogates_and_build_index_map(text: str):
    cleaned_chars = []
    index_map = {}

    new_idx = 0

    for old_idx, char in enumerate(text):
        code = ord(char)

        if 0xD800 <= code <= 0xDFFF:
            continue

        index_map[old_idx] = new_idx
        cleaned_chars.append(char)
        new_idx += 1

    return "".join(cleaned_chars), index_map


def remap_span(start: int, end: int, index_map: dict, cleaned_len: int):
    """
    В DataTurks end обычно inclusive.
    Для модели используем exclusive end.
    """
    if start not in index_map:
        while start not in index_map and start <= end:
            start += 1

    if end not in index_map:
        while end not in index_map and end >= start:
            end -= 1

    if start not in index_map or end not in index_map:
        return None

    new_start = index_map[start]
    new_end = index_map[end] + 1

    if new_start < 0 or new_end > cleaned_len or new_start >= new_end:
        return None

    return new_start, new_end


## Преобразование исходных записей в формат модели


In [11]:
def prepare_examples(raw_data):
    examples = []

    skipped_empty = 0
    skipped_bad_spans = 0
    texts_with_removed_surrogates = 0
    label_counter = Counter()

    for item in raw_data:
        original_text = item.get("text", "")
        annotations = item.get("annotations", [])

        if not original_text or not annotations:
            skipped_empty += 1
            continue

        cleaned_text, index_map = remove_surrogates_and_build_index_map(original_text)

        if len(cleaned_text) != len(original_text):
            texts_with_removed_surrogates += 1

        entities = []

        for ann in annotations:
            if len(ann) != 3:
                skipped_bad_spans += 1
                continue

            start, end, label = ann

            if label not in LABELS_TO_KEEP:
                continue

            span = remap_span(start, end, index_map, len(cleaned_text))

            if span is None:
                skipped_bad_spans += 1
                continue

            new_start, new_end = span
            entity_text = cleaned_text[new_start:new_end]

            if not entity_text.strip():
                skipped_bad_spans += 1
                continue

            entities.append((new_start, new_end, label))
            label_counter[label] += 1

        if entities:
            examples.append(
                {
                    "text": cleaned_text,
                    "entities": entities,
                }
            )

    print("Prepared examples:", len(examples))
    print("Skipped empty:", skipped_empty)
    print("Skipped bad spans:", skipped_bad_spans)
    print("Texts with removed surrogates:", texts_with_removed_surrogates)
    print("Label distribution:")
    print(label_counter)

    return examples


examples = prepare_examples(raw_data)

if MAX_EXAMPLES is not None:
    random.seed(42)
    examples = random.sample(examples, min(MAX_EXAMPLES, len(examples)))

print("Final examples:", len(examples))


Prepared examples: 988
Skipped empty: 0
Skipped bad spans: 0
Texts with removed surrogates: 1
Label distribution:
Counter({'DESIGNATION': 4301, 'LOCATION': 4073, 'PERSON': 3122, 'EDUCATION': 2124, 'COMPANY': 218})
Final examples: 988


## Разбиение длинных резюме на chunks


In [12]:
def split_text_into_chunks_with_entities(
    text: str,
    entities: list,
    max_chars: int = 2000,
    overlap_chars: int = 300,
):
    chunks = []

    text_len = len(text)
    start = 0

    while start < text_len:
        end = min(start + max_chars, text_len)

        if end < text_len:
            next_space = text.rfind(" ", start, end)
            if next_space > start:
                end = next_space

        chunk_text = text[start:end]
        chunk_entities = []

        for ent_start, ent_end, ent_label in entities:
            if ent_start >= start and ent_end <= end:
                chunk_entities.append(
                    {
                        "start": ent_start - start,
                        "end": ent_end - start,
                        "label": ent_label,
                    }
                )

        if chunk_text.strip() and chunk_entities:
            chunks.append(
                {
                    "text": chunk_text,
                    "entities": chunk_entities,
                }
            )

        if end >= text_len:
            break

        start = max(0, end - overlap_chars)

    return chunks


chunked_examples = []

for example in examples:
    chunked_examples.extend(
        split_text_into_chunks_with_entities(
            text=example["text"],
            entities=example["entities"],
            max_chars=CHUNK_MAX_CHARS,
            overlap_chars=CHUNK_OVERLAP_CHARS,
        )
    )

print("Original examples:", len(examples))
print("Chunked examples:", len(chunked_examples))

chunk_label_counter = Counter()
for example in chunked_examples:
    for entity in example["entities"]:
        chunk_label_counter[entity["label"]] += 1

print("Chunk label distribution:")
print(chunk_label_counter)


Original examples: 988
Chunked examples: 2162
Chunk label distribution:
Counter({'DESIGNATION': 4695, 'LOCATION': 4459, 'PERSON': 3321, 'EDUCATION': 2282, 'COMPANY': 221})


## Oversampling chunks с COMPANY


In [13]:
def oversample_company_chunks(chunked_examples, multiplier: int = 5):
    result = []
    company_chunks = []

    for example in chunked_examples:
        result.append(example)

        has_company = any(
            entity["label"] == "COMPANY"
            for entity in example["entities"]
        )

        if has_company:
            company_chunks.append(example)

    for _ in range(multiplier - 1):
        result.extend(company_chunks)

    random.seed(42)
    random.shuffle(result)

    print("Original chunked examples:", len(chunked_examples))
    print("Company chunks:", len(company_chunks))
    print("After oversampling:", len(result))

    label_counter = Counter()
    for example in result:
        for entity in example["entities"]:
            label_counter[entity["label"]] += 1

    print("Label distribution after oversampling:")
    print(label_counter)

    return result


In [14]:
train_chunks_original, dev_examples = train_test_split(
    chunked_examples,
    test_size=0.2,
    random_state=42,
)

train_examples = oversample_company_chunks(
    train_chunks_original,
    multiplier=COMPANY_OVERSAMPLE_MULTIPLIER,
)

print("Train original chunks:", len(train_chunks_original))
print("Train chunks after oversampling:", len(train_examples))
print("Dev chunks:", len(dev_examples))


Original chunked examples: 1729
Company chunks: 151
After oversampling: 2333
Label distribution after oversampling:
Counter({'DESIGNATION': 4512, 'LOCATION': 4101, 'PERSON': 3661, 'EDUCATION': 2436, 'COMPANY': 830})
Train original chunks: 1729
Train chunks after oversampling: 2333
Dev chunks: 433


## Токенизация и перенос entity spans на токены


In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(tokenizer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [16]:
def encode_example(example):
    text = example["text"]
    entities = example["entities"]

    encoding = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )

    offsets = encoding["offset_mapping"]
    labels = [label2id["O"]] * len(offsets)

    for entity in entities:
        ent_start = entity["start"]
        ent_end = entity["end"]
        ent_label = entity["label"]

        token_indices = []

        for idx, (tok_start, tok_end) in enumerate(offsets):
            if tok_start == tok_end:
                continue

            if tok_start < ent_end and tok_end > ent_start:
                token_indices.append(idx)

        if not token_indices:
            continue

        labels[token_indices[0]] = label2id[f"B-{ent_label}"]

        for token_idx in token_indices[1:]:
            labels[token_idx] = label2id[f"I-{ent_label}"]

    encoding.pop("offset_mapping")
    encoding["labels"] = labels

    return encoding


In [17]:
train_dataset = Dataset.from_list(train_examples)
dev_dataset = Dataset.from_list(dev_examples)

train_tokenized = train_dataset.map(
    encode_example,
    remove_columns=train_dataset.column_names,
)

dev_tokenized = dev_dataset.map(
    encode_example,
    remove_columns=dev_dataset.column_names,
)

print(train_tokenized)
print(dev_tokenized)


Map:   0%|          | 0/2333 [00:00<?, ? examples/s]

Map:   0%|          | 0/433 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 2333
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 433
})


## Быстрая проверка разметки после токенизации


In [18]:
sample = train_tokenized[0]

tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])
labels = [
    id2label[label_id] if label_id != -100 else "IGN"
    for label_id in sample["labels"]
]

for token, label in list(zip(tokens, labels))[:150]:
    if label != "O" and label != "IGN":
        print(token, label)


k B-PERSON
##ima I-PERSON
##ya I-PERSON
son I-PERSON
##awa I-PERSON
##ne I-PERSON
Than B-LOCATION
##e I-LOCATION
, I-LOCATION
Technical B-DESIGNATION
Support I-DESIGNATION
Engineer I-DESIGNATION
SA B-COMPANY
##P I-COMPANY
- I-COMPANY
2016 B-EDUCATION
B B-EDUCATION
##E I-EDUCATION
in I-EDUCATION
computer I-EDUCATION
science I-EDUCATION
SS B-PERSON
##VP I-PERSON
##S I-PERSON
’ I-PERSON
s I-PERSON
Late I-PERSON
B I-PERSON
. I-PERSON
S I-PERSON
. I-PERSON
De I-PERSON
##ore I-PERSON
College I-PERSON
of I-PERSON
Engineering I-PERSON


## Инициализация модели


In [19]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

model.to(device)

print("Num labels:", len(label_list))


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Num labels: 11


## Метрики качества

In [20]:
seqeval = evaluate.load("seqeval")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue

            current_preds.append(id2label[int(pred_id)])
            current_labels.append(id2label[int(label_id)])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels,
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


## Обучение модели


In [21]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=N_EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)


In [22]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=dev_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [23]:
trainer.train()


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.158645,0.156176,0.304315,0.426323,0.355132,0.951615
2,0.135018,0.142139,0.342093,0.526109,0.414600,0.953934
3,0.094750,0.132925,0.387870,0.544349,0.452976,0.958235
4,0.089727,0.123497,0.400369,0.542561,0.460744,0.961017
5,0.084379,0.126505,0.406639,0.560801,0.471437,0.960404


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1460, training_loss=0.13773971285722028, metrics={'train_runtime': 1279.5897, 'train_samples_per_second': 9.116, 'train_steps_per_second': 1.141, 'total_flos': 3022811485372860.0, 'train_loss': 0.13773971285722028, 'epoch': 5.0})

## Оценка модели


In [24]:
metrics = trainer.evaluate()

metrics


{'eval_loss': 0.1265053004026413,
 'eval_precision': 0.4066390041493776,
 'eval_recall': 0.5608011444921316,
 'eval_f1': 0.47143716175586287,
 'eval_accuracy': 0.9604039505935502,
 'eval_runtime': 14.5398,
 'eval_samples_per_second': 29.78,
 'eval_steps_per_second': 3.783,
 'epoch': 5.0}

## Сохранение модели и метрик


In [25]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

metrics_path = REPORT_DIR / "resume_bert_ner_chunked_metrics.json"

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

metadata = {
    "model_type": "bert_token_classification_ner",
    "base_model": MODEL_NAME,
    "labels_to_keep": LABELS_TO_KEEP,
    "label_list": label_list,
    "max_length": MAX_LENGTH,
    "chunk_max_chars": CHUNK_MAX_CHARS,
    "chunk_overlap_chars": CHUNK_OVERLAP_CHARS,
    "company_oversample_multiplier": COMPANY_OVERSAMPLE_MULTIPLIER,
    "metrics": metrics,
}

metadata_path = MODEL_DIR / "metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Saved model to:", MODEL_DIR)
print("Saved metrics to:", metrics_path)
print("Saved metadata to:", metadata_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: app/models/resume_bert_ner_model_chunked
Saved metrics to: experiments/reports/resume_bert_ner_chunked_metrics.json
Saved metadata to: app/models/resume_bert_ner_model_chunked/metadata.json


## Проверка инференса


In [26]:
ner_pipeline = pipeline(
    "token-classification",
    model=str(MODEL_DIR),
    tokenizer=str(MODEL_DIR),
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1,
)


def print_bert_ner(text: str):
    entities = ner_pipeline(text[:3000])

    if not entities:
        print("No entities found")
        return

    for ent in entities:
        print(
            ent["word"],
            "->",
            ent["entity_group"],
            "| score:",
            round(float(ent["score"]), 4),
        )


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [27]:
test_text_data_engineer = """
John Smith
Senior Data Engineer at ABC Analytics Company
Email: john.smith@gmail.com
Phone: +44 7700 900123
Location: London, United Kingdom

Summary
Data Engineer with 5 years of experience in building data pipelines and analytical platforms.

Work Experience
Senior Data Engineer
ABC Analytics Company
January 2021 to Present

Developed ETL pipelines using Python, SQL, Apache Spark, PySpark, Airflow, Hadoop, Hive and Kafka.

Education
Master's Degree in Computer Science
University of London

Skills
Python, SQL, Spark, PySpark, Airflow, Hadoop, Hive, Kafka, Docker, PostgreSQL
"""

print_bert_ner(test_text_data_engineer)


John Smith -> PERSON | score: 0.9258
Senior Data Engineer -> DESIGNATION | score: 0.9091
Analytics Company -> COMPANY | score: 0.5918
London, United Kingdom -> LOCATION | score: 0.9055
Data Engineer -> DESIGNATION | score: 0.5833
Master ' s Degree in Computer Science -> EDUCATION | score: 0.9688
University of London -> PERSON | score: 0.8777


In [28]:
test_text_qa = """
Michael Brown
QA Automation Engineer at QualityWorks Ltd
Email: michael.brown@testmail.com
Phone: +49 160 1234567
Location: Berlin, Germany

Summary
QA Engineer with experience in manual testing, automation testing, API testing and regression testing.

Work Experience
QA Automation Engineer
QualityWorks Ltd
June 2019 to Present

Created automated test cases using Selenium, Python and Pytest.
Tested REST APIs using Postman and prepared test documentation in Jira and Confluence.

Education
Bachelor of Information Technology
Technical University of Berlin
"""

print_bert_ner(test_text_qa)


Michael Brown -> PERSON | score: 0.9398
QA Automation Engineer -> DESIGNATION | score: 0.9159
##Works Ltd -> COMPANY | score: 0.7753
Berlin, Germany -> LOCATION | score: 0.8841
Q -> DESIGNATION | score: 0.5208
##mation Engineer -> DESIGNATION | score: 0.5992
Bachelor of Information Technology -> EDUCATION | score: 0.9734
Technical University of Berlin -> PERSON | score: 0.8194


In [29]:
test_text_ba = """
Anna Petrova
IT Business Analyst at FinCore Bank
Email: anna.petrova@fincore.com
Phone: +371 2555 8899
Location: Riga, Latvia

Summary
IT Business Analyst with experience in requirements gathering, business process modeling and software development projects.

Work Experience
IT Business Analyst
FinCore Bank
February 2021 to Present

Collected business requirements from stakeholders and prepared user stories.
Created BPMN diagrams, UML diagrams, functional specifications and acceptance criteria.

Education
Master's Degree in Business Informatics
University of Latvia
"""

print_bert_ner(test_text_ba)


Anna Petrova -> PERSON | score: 0.9049
IT Business Analyst -> DESIGNATION | score: 0.9629
FinCore Bank -> COMPANY | score: 0.7973
Riga, Latvia -> LOCATION | score: 0.7328
IT Business Analyst -> DESIGNATION | score: 0.6489
Master ' s Degree in Business Informatics -> EDUCATION | score: 0.9541
University of Latvia -> PERSON | score: 0.8537
